## بسم الله الرحمن الرحیم

### نسخه پکیج‌ها، GPU و مسیرها را چک می کنیم

# 00 | Setup Check

این نوت‌بوک محیط اجرا، مسیرها، پکیج‌های نصب‌شده و سلامت فایل‌های ضروری را بررسی می‌کند.

**قبل از اجرا مطمئن شوید:**
- پکیج‌های `nibabel`, `numpy`, `scipy`, `pandas`, `matplotlib`, `medim`, `torch` نصب هستند.
- پوشه رسمی `SAM_Med3D` در مسیر کنار پروژه وجود دارد.
- چک‌پوینت `sam_med3d_turbo.pth` در `SAM_Med3D/ckpt` قرار دارد.

In [1]:
import sys
import platform
from pathlib import Path

print("Python version:", sys.version)
print("Platform:", platform.platform())

# مسیر پروژه را تشخیص بده
# اگر نوت‌بوک داخل ACDC-SAMMed3D/notebooks باشد:
CURRENT = Path.cwd()
if CURRENT.name == "notebooks" and CURRENT.parent.name == "ACDC-SAMMed3D":
    PROJECT_ROOT = CURRENT.parent
else:
    # اگر در پوشه دیگری اجرا می‌کنید، مسیر را دستی ست کنید
    PROJECT_ROOT = Path(r"C:\Users\MahdiCS313\Downloads\Proposal_Nazanin\ACDC-SAMMed3D")

OFFICIAL_REPO = PROJECT_ROOT.parent / "SAM_Med3D"
RAW_DATA = PROJECT_ROOT / "data" / "ACDC" / "database"
OUTPUT_ROOT = PROJECT_ROOT.parent / "outputs"
TASK_DIR = OUTPUT_ROOT / "Task010_ACDC"

print("\nPROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DATA     :", RAW_DATA)
print("OFFICIAL_REPO:", OFFICIAL_REPO)
print("OUTPUT_ROOT  :", OUTPUT_ROOT)
print("TASK_DIR     :", TASK_DIR)

Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
Platform: Windows-10-10.0.19045-SP0

PROJECT_ROOT : c:\Users\MahdiCS313\Downloads\Proposal_Nazanin\ACDC-SAMMed3D
RAW_DATA     : c:\Users\MahdiCS313\Downloads\Proposal_Nazanin\ACDC-SAMMed3D\data\ACDC\database
OFFICIAL_REPO: c:\Users\MahdiCS313\Downloads\Proposal_Nazanin\SAM_Med3D
OUTPUT_ROOT  : c:\Users\MahdiCS313\Downloads\Proposal_Nazanin\outputs
TASK_DIR     : c:\Users\MahdiCS313\Downloads\Proposal_Nazanin\outputs\Task010_ACDC


In [2]:
import importlib

required_packages = [
    "numpy", "nibabel", "scipy", "pandas", "matplotlib",
    "torch", "torchio", "monai", "medim", "SimpleITK"
]

print("Checking installed packages...\n")
for pkg in required_packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "unknown")
        print(f"[OK] {pkg:15s} version={version}")
    except ImportError as e:
        print(f"[MISSING] {pkg:15s} -> {e}")

# Check CUDA
try:
    import torch
    print("\nCUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device count:", torch.cuda.device_count())
        print("Current device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch CUDA check failed:", e)

Checking installed packages...

[OK] numpy           version=2.2.6
[OK] nibabel         version=5.4.2
[OK] scipy           version=1.15.3
[OK] pandas          version=2.3.3
[OK] matplotlib      version=3.10.9
[OK] torch           version=2.7.1+cu118
[MISSING] torchio         -> No module named 'torchio'


c:\Users\MahdiCS313\miniconda3\envs\medical-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] monai           version=1.6.0
[OK] medim           version=unknown
[OK] SimpleITK       version=2.5.6

CUDA available: True
CUDA device count: 1
Current device: NVIDIA GeForce GTX 1060


In [3]:
print("Checking raw data directory...")
training_dir = RAW_DATA / "training"
testing_dir = RAW_DATA / "testing"

print("Training exists:", training_dir.exists())
print("Testing exists:", testing_dir.exists())

if training_dir.exists():
    patients = sorted([p for p in training_dir.iterdir() if p.is_dir() and p.name.startswith("patient")])
    print(f"Training patients: {len(patients)}")
    if patients:
        first_patient = patients[0]
        nii_files = sorted(first_patient.glob("*.nii"))
        print(f"First patient: {first_patient.name}")
        for f in nii_files:
            print("   ", f.name)

Checking raw data directory...
Training exists: True
Testing exists: True
Training patients: 100
First patient: patient001
    patient001_4d.nii
    patient001_frame01.nii
    patient001_frame01_gt.nii
    patient001_frame12.nii
    patient001_frame12_gt.nii


In [4]:
print("Checking official repo...")
for fname in ["train.py", "medim_val_dataset.py", "medim_val_single.py", "train.sh", "val_on_dataset.sh"]:
    f = OFFICIAL_REPO / fname
    if f.exists():
        print(f"[OK] {fname}")
    else:
        print(f"[MISSING] {fname}")

ckpt = OFFICIAL_REPO / "ckpt" / "sam_med3d_turbo.pth"
print("\nCheckpoint turbo:", ckpt.exists())
if ckpt.exists():
    print("Checkpoint size (MB):", ckpt.stat().st_size / (1024*1024))

Checking official repo...
[OK] train.py
[OK] medim_val_dataset.py
[OK] medim_val_single.py
[OK] train.sh
[OK] val_on_dataset.sh

Checkpoint turbo: True
Checkpoint size (MB): 383.5331211090088


In [5]:
print("Setup check completed.")

Setup check completed.
